In [4]:
"""
MSPE Ratios and Rolling Window Charts
======================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import os
import warnings
warnings.filterwarnings('ignore')

INPUT_FILE   = 'Input_All_Forecasts_Combined.xlsx'
OUTPUT_EXCEL = 'Output_Forecast_Analysis.xlsx'
BENCHMARK    = 'RW_avg (benchmark)'
HORIZONS     = [1, 3, 6, 9, 12, 15, 18, 21, 24]
WINDOW       = 24

script_dir = os.getcwd()

print("Loading data...")
df = pd.read_excel(os.path.join(script_dir, INPUT_FILE))
df['forecast_origin'] = pd.to_datetime(df['forecast_origin'])
df = df.dropna(subset=['actual'])
print(f"Rows: {len(df):,}  |  Models: {df['model'].nunique()}  |  "
      f"Origins: {df['forecast_origin'].nunique()}")

ALL_MODELS = sorted(df['model'].unique().tolist())
print(f"  Models found: {ALL_MODELS}")

df['sq_error'] = (df['forecast'] - df['actual']) ** 2
print("  No exclusions — raw MSPE reported.")

print("Computing MSPE ratios...")
mspe_raw = (df.groupby(['model', 'horizon'])['sq_error']
              .mean()
              .reset_index()
              .rename(columns={'sq_error': 'mspe'}))

bench = (mspe_raw[mspe_raw['model'] == BENCHMARK]
         [['horizon', 'mspe']]
         .rename(columns={'mspe': 'mspe_bench'}))

mspe_raw  = mspe_raw.merge(bench, on='horizon')
mspe_raw['ratio'] = mspe_raw['mspe'] / mspe_raw['mspe_bench']

mspe_wide = (mspe_raw.pivot(index='model', columns='horizon', values='ratio')
                     .reindex(columns=HORIZONS))
mspe_wide['Average'] = mspe_wide.mean(axis=1)
mspe_wide = mspe_wide.sort_values('Average')

print("  Top 10 models by average MSPE ratio:")
for m, row in mspe_wide.head(10).iterrows():
    print(f"    {m:45s}  avg={row['Average']:.3f}")

print(f"Computing {WINDOW}-month rolling MSPE ratios...")
monthly_ratio = {}
for h in HORIZONS:
    sub = df[df['horizon'] == h].sort_values('forecast_origin')
    bench_roll = (sub[sub['model'] == BENCHMARK]
                  .set_index('forecast_origin')['sq_error']
                  .rolling(WINDOW).mean()
                  .rename('bench_mspe'))
    roll_dict = {}
    for m in ALL_MODELS:
        mod_roll = (sub[sub['model'] == m]
                    .set_index('forecast_origin')['sq_error']
                    .rolling(WINDOW).mean())
        roll_dict[m] = mod_roll / bench_roll
    monthly_ratio[h] = pd.DataFrame(roll_dict).dropna(how='all')
print("  Done.")

print("Creating rolling MSPE figures...")
cmap   = plt.cm.get_cmap('tab20', len(ALL_MODELS))
COLORS = {m: cmap(i) for i, m in enumerate(ALL_MODELS)}
COLORS[BENCHMARK] = '#000000'

for h in HORIZONS:
    fig, ax = plt.subplots(figsize=(14, 6))
    mdf = monthly_ratio[h]
    for m in ALL_MODELS:
        if m not in mdf.columns:
            continue
        s = mdf[m].dropna()
        ax.plot(s.index, s.values, color=COLORS[m], linewidth=2,
                linestyle='-', alpha=0.75, label=m)
    ax.axhline(1.0, color='black', linewidth=1.5, linestyle='--')
    ax.set_title(f'{WINDOW}-Month Rolling MSPE Ratio Relative to RW Benchmark  |  h = {h}',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('MSPE ratio (model / RW benchmark)', fontsize=9)
    ax.set_xlabel('Forecast origin', fontsize=9)
    ax.set_ylim(0, 5)
    ax.grid(alpha=0.2, linestyle='--')
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=6, ncol=3, loc='upper right', frameon=True, framealpha=0.8)
    fname = os.path.join(script_dir, f'fig_rolling_mspe_h{h}.png')
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: fig_rolling_mspe_h{h}.png")

print("Creating MSPE bar chart...")
models_in_wide = [m for m in mspe_wide.index if m != BENCHMARK]
x       = np.arange(len(HORIZONS))
width   = 0.8 / len(models_in_wide)
offsets = np.linspace(-(len(models_in_wide)-1)/2*width,
                       (len(models_in_wide)-1)/2*width,
                       len(models_in_wide))
fig2, ax2 = plt.subplots(figsize=(16, 7))
for i, m in enumerate(models_in_wide):
    vals = mspe_wide.loc[m, HORIZONS].values.astype(float)
    ax2.bar(x + offsets[i], vals, width=width*0.9,
            color=COLORS.get(m, '#888'), alpha=0.8, label=m)
ax2.axhline(1.0, color='black', linewidth=1.5, linestyle='--', label='Benchmark = 1')
ax2.set_xticks(x)
ax2.set_xticklabels([f'h={h}' for h in HORIZONS])
ax2.set_ylabel('MSPE ratio relative to RW benchmark', fontsize=10)
ax2.set_title('MSPE Ratios by Horizon - All Models', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 3)
ax2.grid(alpha=0.2, axis='y', linestyle='--')
ax2.legend(fontsize=6, ncol=3, loc='upper right', frameon=True, framealpha=0.8)
plt.tight_layout()
plt.savefig(os.path.join(script_dir, 'fig_mspe_bar_all_models.png'), dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: fig_mspe_bar_all_models.png")

print("Building Excel...")
BLUE  = PatternFill('solid', fgColor='1F4E79')
DBLUE = PatternFill('solid', fgColor='2E75B6')
LBLUE = PatternFill('solid', fgColor='BDD7EE')
GREEN = PatternFill('solid', fgColor='C6EFCE')
RED   = PatternFill('solid', fgColor='FFC7CE')
GREY  = PatternFill('solid', fgColor='F2F2F2')
WHITE = PatternFill('solid', fgColor='FFFFFF')
CLASS_FILLS = {
    'Benchmark':     PatternFill('solid', fgColor='FFF2CC'),
    'Univariate':    PatternFill('solid', fgColor='E2EFDA'),
    'Multivariate':  PatternFill('solid', fgColor='DEEAF1'),
    'Cointegration': PatternFill('solid', fgColor='F4CCCC'),
    'Price Spread':  PatternFill('solid', fgColor='EAD1DC'),
    'Market-based':  PatternFill('solid', fgColor='D9D2E9'),
}

def sw(ws, widths):
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

def c(ws, r, col, v, fill=WHITE, bold=False, fmt=None,
      align='center', color='000000', cs=1):
    if cs > 1:
        ws.merge_cells(start_row=r, start_column=col,
                       end_row=r, end_column=col+cs-1)
    cl = ws.cell(r, col, v)
    cl.font      = Font(bold=bold, size=10, color=color)
    cl.fill      = fill
    cl.alignment = Alignment(horizontal=align, vertical='center', wrap_text=True)
    if fmt:
        cl.number_format = fmt

wb = openpyxl.Workbook()

ws1 = wb.active
ws1.title = 'MSPE Ratios'
sw(ws1, [38] + [9]*9 + [9])
c(ws1, 1, 1,
  f'MSPE Ratios Relative to {BENCHMARK}  |  ratio < 1 = better than benchmark',
  fill=BLUE, bold=True, color='FFFFFF', cs=11)
ws1.row_dimensions[1].height = 28
c(ws1, 2, 1,
  (f'Jan 2015 - Nov 2025  |  131 origins  |  '
   f'Raw MSPE - no forecasts excluded  |  '
   f'Green < 0.85  |  Red > 1.15  |  Bold = beats benchmark'),
  fill=LBLUE, align='left', cs=11)
ws1.row_dimensions[2].height = 18
c(ws1, 3, 1, 'Model', fill=DBLUE, bold=True, color='FFFFFF', align='left')
for ci, h in enumerate(HORIZONS, 2):
    c(ws1, 3, ci, f'h={h}', fill=DBLUE, bold=True, color='FFFFFF')
c(ws1, 3, 11, 'Average', fill=DBLUE, bold=True, color='FFFFFF')
ws1.row_dimensions[3].height = 20
for ri, (model_name, row) in enumerate(mspe_wide.iterrows()):
    r   = ri + 4
    alt = GREY if ri % 2 == 0 else WHITE
    is_bench = (model_name == BENCHMARK)
    c(ws1, r, 1, model_name, fill=LBLUE if is_bench else alt, bold=is_bench, align='left')
    for ci, h in enumerate(HORIZONS, 2):
        v = row[h]
        if np.isnan(v):
            c(ws1, r, ci, '—', fill=alt)
            continue
        f = GREEN if v < 0.85 else (RED if v > 1.15 else alt)
        c(ws1, r, ci, round(v, 3), fill=f, bold=(v < 1.0), fmt='0.000')
    avg   = row['Average']
    f_avg = GREEN if avg < 0.85 else (RED if avg > 1.15 else alt)
    c(ws1, r, 11, round(avg, 3) if not np.isnan(avg) else '—',
      fill=f_avg, bold=(avg < 1.0), fmt='0.000')
ws1.freeze_panes = 'A4'

ws2 = wb.create_sheet('By Model Class')
sw(ws2, [18, 38] + [9]*9 + [9])
c(ws2, 1, 1, 'MSPE Ratios by Model Class', fill=BLUE, bold=True, color='FFFFFF', cs=12)
ws2.row_dimensions[1].height = 28
c(ws2, 2, 1, 'Class', fill=DBLUE, bold=True, color='FFFFFF', align='left')
c(ws2, 2, 2, 'Model', fill=DBLUE, bold=True, color='FFFFFF', align='left')
for ci, h in enumerate(HORIZONS, 3):
    c(ws2, 2, ci, f'h={h}', fill=DBLUE, bold=True, color='FFFFFF')
c(ws2, 2, 12, 'Average', fill=DBLUE, bold=True, color='FFFFFF')
ws2.row_dimensions[2].height = 20
model_classes = {
    'Benchmark':     ['RW_avg (benchmark)', 'RW_EOM', 'RW_drift'],
    'Univariate':    ['AR(1)', 'AR(12)', 'AR(AIC,p≤6)',
                      'BAR(1)', 'BAR(12)', 'BAR(AIC,p≤6)',
                      'ARMA(1,1)', 'IMA(1,1)', 'ARIMA(1,1,1)',
                      'Exponential Smoothing (alpha=0.80)'],
    'Multivariate':  ['VAR(1)', 'VAR(AIC,p<=6)', 'BVAR(1)', 'BVAR(AIC,p<=6)'],
    'Cointegration': ['VEC(1)', 'VEC(12)', 'VEC(AIC,p<=6)',
                      'BVEC(1)', 'BVEC(12)', 'BVEC(AIC,p<=6)'],
    'Price Spread':  ['PriceSpread(40) α̂,β̂',
                      'PriceSpread(41) α=0,β̂',
                      'PriceSpread(42) MS alpha_MS beta_MS'],
    'Market-based':  ['futures'],
}
r = 3
for cls, models in model_classes.items():
    cfill = CLASS_FILLS[cls]
    first = True
    for m in models:
        if m not in mspe_wide.index:
            continue
        row = mspe_wide.loc[m]
        c(ws2, r, 1, cls if first else '', fill=cfill, bold=first, align='left')
        c(ws2, r, 2, m, fill=cfill, align='left')
        for ci, h in enumerate(HORIZONS, 3):
            v = row[h]
            f = GREEN if v < 0.85 else (RED if v > 1.15 else cfill)
            c(ws2, r, ci, round(v, 3) if not np.isnan(v) else '—',
              fill=f, bold=(v < 1.0), fmt='0.000')
        avg   = row['Average']
        f_avg = GREEN if avg < 0.85 else (RED if avg > 1.15 else cfill)
        c(ws2, r, 12, round(avg, 3) if not np.isnan(avg) else '—',
          fill=f_avg, bold=(avg < 1.0), fmt='0.000')
        first = False
        r += 1
    r += 1
ws2.freeze_panes = 'A3'

ws3 = wb.create_sheet('Rolling MSPE Ratios')
c(ws3, 1, 1,
  f'{WINDOW}-Month Rolling MSPE Ratios - rolling(model MSPE) / rolling(benchmark MSPE)',
  fill=BLUE, bold=True, color='FFFFFF', cs=len(ALL_MODELS)+2)
ws3.row_dimensions[1].height = 25
row_ptr = 3
for h in HORIZONS:
    ws3.merge_cells(start_row=row_ptr, start_column=1,
                    end_row=row_ptr, end_column=len(ALL_MODELS)+1)
    hc = ws3.cell(row_ptr, 1, f'Horizon h = {h}')
    hc.font      = Font(bold=True, size=11, color='1F4E79')
    hc.fill      = LBLUE
    hc.alignment = Alignment(horizontal='left', vertical='center')
    row_ptr += 1
    c(ws3, row_ptr, 1, 'Forecast Origin', fill=DBLUE, bold=True, color='FFFFFF')
    for ci, m in enumerate(ALL_MODELS, 2):
        c(ws3, row_ptr, ci, m[:24], fill=DBLUE, bold=True, color='FFFFFF')
    row_ptr += 1
    mdf = monthly_ratio[h]
    for date, datarow in mdf.iterrows():
        c(ws3, row_ptr, 1, date.strftime('%Y-%m'))
        for ci, m in enumerate(ALL_MODELS, 2):
            v = datarow.get(m, np.nan)
            c(ws3, row_ptr, ci, round(float(v), 4) if not np.isnan(v) else '', fmt='0.0000')
        row_ptr += 1
    row_ptr += 2
ws3.column_dimensions['A'].width = 16
for ci in range(2, len(ALL_MODELS)+2):
    ws3.column_dimensions[get_column_letter(ci)].width = 14
ws3.freeze_panes = 'A3'

wb.save(os.path.join(script_dir, OUTPUT_EXCEL))
print(f"  Saved: {OUTPUT_EXCEL}")
print("Complete.")
print(f"  {OUTPUT_EXCEL}")
print(f"  fig_rolling_mspe_h1.png ... fig_rolling_mspe_h24.png")
print(f"  fig_mspe_bar_all_models.png")


Loading data...
Rows: 29,133  |  Models: 27  |  Origins: 131
  Models found: ['AR(1)', 'AR(12)', 'AR(AIC,p≤6)', 'ARIMA(1,1,1)', 'ARMA(1,1)', 'BAR(1)', 'BAR(12)', 'BAR(AIC,p≤6)', 'BVAR(1)', 'BVAR(AIC,p<=6)', 'BVEC(1)', 'BVEC(12)', 'BVEC(AIC,p<=6)', 'Exponential Smoothing (alpha=0.80)', 'IMA(1,1)', 'PriceSpread(40) α̂,β̂', 'PriceSpread(41) α=0,β̂', 'PriceSpread(42) MS alpha_MS beta_MS', 'RW_EOM', 'RW_avg (benchmark)', 'RW_drift', 'VAR(1)', 'VAR(AIC,p<=6)', 'VEC(1)', 'VEC(12)', 'VEC(AIC,p<=6)', 'futures']
  No exclusions — raw MSPE reported.
Computing MSPE ratios...
  Top 10 models by average MSPE ratio:
    AR(12)                                         avg=0.684
    BAR(12)                                        avg=0.685
    VEC(12)                                        avg=0.763
    ARMA(1,1)                                      avg=0.768
    PriceSpread(41) α=0,β̂                         avg=0.819
    PriceSpread(40) α̂,β̂                          avg=0.823
    PriceSpread(42) MS al